# Logistic Regression — C00~C03 최종 4조합 확인

`rollup16/rollup × lsvd/lnmf` 2×2 조합을 SKF와 Group5에서 확인한다.
각 후보는 자신의 outer-train 내부에서 C를 별도로 선택하고 outer validation은
모델 선택에 사용하지 않는다.

주의: `rollup`에는 fe25 25개가 이미 들어 있다. 요청한 조합을 그대로 실행하므로
C01/C03은 fe25 25개가 중복되고, C00/C02는 rollup16과 fe25 중 9개가 중복된다.
결과표와 manifest에 이 중복 수를 함께 기록한다.

In [ ]:
from __future__ import annotations

import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = Path("/kaggle/input").is_dir()

REPO_URL = "https://github.com/cancer-classification-ai/onco-ai.git"
# 2026-08-05 현재 develop의 train_linear.py는 빈 파일이다.
# 이 branch가 develop에 merge된 뒤에는 "develop"으로 바꿔도 된다.
PROJECT_BRANCH = "codex/logistic-shared-tabular-pipeline"

# "kagglehub": Kaggle Dataset slug에서 다운로드
# "drive": DRIVE_DATA_DIR에 이미 올려둔 CSV 3개 사용
DATA_SOURCE = "kagglehub"
KAGGLE_DATASET = "hyunwoo11/onco-data-hack"
DRIVE_DATA_DIR = Path("/content/drive/MyDrive/onco-data-hack")
DATA_ROOT = ""  # 아래 데이터 셀이 DATA_SOURCE에 따라 채운다.
MOUNT_GOOGLE_DRIVE = True
INSTALL_CORE_PACKAGES = True

SEED = 42
N_SPLITS = 5
CVS = ("skf", "sgkf")  # sgkf가 fold_group5
TOPK = 500
C_GRID = [0.001, 0.003, 0.01, 0.03, 0.1]
INNER_VALID_FRACTION = 0.15
SOLVER = "lbfgs"
SCALER = "standard"
MAX_ITER = 1000
TOL = 1e-4
TAG = "logreg_final4_confirmation_v1"

if IN_COLAB:
    WORK_ROOT = Path("/content")
elif IN_KAGGLE:
    WORK_ROOT = Path("/kaggle/working")
else:
    WORK_ROOT = Path.cwd()
KAGGLE_DOWNLOAD_DIR = WORK_ROOT / "onco-data-hack"

REPO_DIR = WORK_ROOT / "onco-ai-logreg"
if Path.cwd().name == "onco-ai" and (Path.cwd() / "scripts").is_dir():
    REPO_DIR = Path.cwd()

if IN_COLAB:
    OUTPUT_ROOT = Path("/content/drive/MyDrive/onco_logreg_final4_confirmation")
elif IN_KAGGLE:
    OUTPUT_ROOT = Path("/kaggle/working/onco_logreg_final4_confirmation")
else:
    OUTPUT_ROOT = REPO_DIR / "artifacts" / "logreg_final4_confirmation"

print("environment :", "Colab" if IN_COLAB else "Kaggle" if IN_KAGGLE else "local")
print("repo        :", REPO_DIR)
print("output      :", OUTPUT_ROOT)
print("device      : CPU (GPU는 사용하지 않음)")



## 1. 저장소와 패키지 준비

이미 clone되어 있으면 덮어쓰거나 pull하지 않는다. 출력된 commit SHA를 결과
manifest에도 기록한다.



In [ ]:
if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

if not (REPO_DIR / ".git").is_dir():
    subprocess.run(
        [
            "git",
            "clone",
            "--branch",
            PROJECT_BRANCH,
            "--single-branch",
            REPO_URL,
            str(REPO_DIR),
        ],
        check=True,
    )

if INSTALL_CORE_PACKAGES:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "numpy",
            "pandas",
            "scipy",
            "scikit-learn",
            "pyarrow",
            "PyYAML",
            "kagglehub",
        ],
        check=True,
    )

os.chdir(REPO_DIR)
commit_sha = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()
branch_name = subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip()
print("branch      :", branch_name)
print("commit      :", commit_sha)

linear_driver = REPO_DIR / "scripts/train_linear.py"
linear_model = REPO_DIR / "src/cancer_hack/models_linear.py"
if not linear_driver.is_file() or linear_driver.stat().st_size < 1000 or not linear_model.is_file():
    raise RuntimeError(
        "fold-safe Logistic 코드가 없는 branch입니다. "
        f"PROJECT_BRANCH={PROJECT_BRANCH!r}를 사용하거나 해당 branch를 develop에 merge하세요."
    )



## 2. train/test/process 위치 확인

`DATA_SOURCE="kagglehub"`이면 Dataset을 다운로드한 뒤 반환된 실제 경로를 사용한다.
비공개 Dataset은 Colab Secrets에 `KAGGLE_API_TOKEN`이 필요하다. `drive`이면
`DRIVE_DATA_DIR`를 그대로 사용한다.

지원 layout:

- `<DATA_ROOT>/raw` + `<DATA_ROOT>/process`
- `<DATA_ROOT>/data/raw` + `<DATA_ROOT>/data/process`
- `<DATA_ROOT>`에 CSV가 직접 있고 `<DATA_ROOT>/process`에 parquet

process 파일이 Dataset에 있으면 작업 폴더에 symlink하고, 누락분만 다음 셀에서 만든다.



In [ ]:
def valid_raw(path: Path) -> bool:
    return all((path / name).is_file() for name in ("train.csv", "test.csv"))


def resolve_data_layout(explicit: str) -> tuple[Path, Path | None]:
    roots: list[Path] = []
    if explicit:
        roots.append(Path(explicit).expanduser())
    if IN_KAGGLE:
        roots.extend(sorted(Path("/kaggle/input").glob("*")))
        roots.extend(sorted(Path("/kaggle/input").glob("*/*/*")))
    roots.extend(
        [
            REPO_DIR / "data",
            Path("/content/onco-ai/data"),  # 기존 LightGBM Colab clone
            Path("/content/onco-ai"),
            Path("/content/data"),
            Path("/content"),  # Colab 파일 패널에 CSV를 직접 업로드한 경우
            Path("/content/onco-data-hack"),
            Path("/content/drive/MyDrive/onco-data-hack"),
        ]
    )
    checked = []
    for root in roots:
        layouts = [
            (root / "raw", root / "process"),
            (root / "data/raw", root / "data/process"),
            (root, root / "process"),
        ]
        for raw, process in layouts:
            checked.append(str(raw))
            if valid_raw(raw):
                return raw.resolve(), process.resolve() if process.is_dir() else None

    # 폴더 이름이 다른 경우 /content 아래 최대 4단계에서 train.csv를 찾는다.
    discovered = []
    content = Path("/content")
    if content.is_dir():
        for pattern in (
            "train.csv",
            "*/train.csv",
            "*/*/train.csv",
            "*/*/*/train.csv",
            "*/*/*/*/train.csv",
        ):
            discovered.extend(content.glob(pattern))
    for train_path in sorted(set(discovered)):
        raw = train_path.parent
        if not valid_raw(raw):
            continue
        process_candidates = (raw.parent / "process", raw / "process")
        process = next((path for path in process_candidates if path.is_dir()), None)
        return raw.resolve(), process.resolve() if process is not None else None
    raise FileNotFoundError(
        "train.csv/test.csv가 현재 Colab에 없습니다. Drive/업로드 위치를 "
        "DATA_ROOT에 지정하세요.\n확인한 raw 후보:\n"
        + "\n".join(checked[:40])
        + "\n발견한 train.csv:\n"
        + ("\n".join(map(str, discovered)) if discovered else "(없음)")
    )


if DATA_SOURCE == "kagglehub":
    # Colab Secrets(열쇠 아이콘)에 KAGGLE_API_TOKEN을 등록하면 자동 사용한다.
    if IN_COLAB and not os.environ.get("KAGGLE_API_TOKEN"):
        try:
            from google.colab import userdata

            token = userdata.get("KAGGLE_API_TOKEN")
            if token:
                os.environ["KAGGLE_API_TOKEN"] = token
        except Exception:
            pass
    import kagglehub

    try:
        downloaded = Path(
            kagglehub.dataset_download(
                KAGGLE_DATASET,
                output_dir=str(KAGGLE_DOWNLOAD_DIR),
            )
        )
    except Exception as exc:
        raise RuntimeError(
            "Kaggle Dataset 다운로드 실패. Colab Secrets에 KAGGLE_API_TOKEN을 "
            "등록했는지 확인하세요. 403이면 Kaggle Dataset 페이지에서 필요한 "
            "동의/접근 권한도 먼저 완료해야 합니다."
        ) from exc
    DATA_ROOT = str(downloaded)
elif DATA_SOURCE == "drive":
    DATA_ROOT = str(DRIVE_DATA_DIR)
else:
    raise ValueError("DATA_SOURCE는 'kagglehub' 또는 'drive'여야 합니다.")

print("selected data source:", DATA_SOURCE, DATA_ROOT)
RAW_SOURCE, PROCESS_SOURCE = resolve_data_layout(DATA_ROOT)
PROCESS_DIR = REPO_DIR / "data/process"
PROCESS_DIR.mkdir(parents=True, exist_ok=True)

if PROCESS_SOURCE is not None and PROCESS_SOURCE != PROCESS_DIR.resolve():
    for source in PROCESS_SOURCE.iterdir():
        if not source.is_file():
            continue
        target = PROCESS_DIR / source.name
        if not target.exists():
            target.symlink_to(source)

required_raw = ["train.csv", "test.csv", "sample_submission.csv"]
missing_raw = [name for name in required_raw if not (RAW_SOURCE / name).is_file()]
if missing_raw:
    raise FileNotFoundError(f"RAW_SOURCE에 없는 파일: {missing_raw}")

print("raw source    :", RAW_SOURCE)
print("process source:", PROCESS_SOURCE)
print("process work  :", PROCESS_DIR)
print("existing parquet:", len(list(PROCESS_DIR.glob("*.parquet"))))



## 3. 누락된 feature parquet만 생성

이번 조합에서 필요한 `domain`, `rollup/fe25`, `enc3`, `ptok`, `ebovr` 원본만 만든다.
`lsvd`, `lnmf`, `csig`, `ebovr`는 각 outer fold 안에서 fit/transform한다.

In [ ]:
FEATURE_COMMANDS = {
    "domain_features.parquet": ["domain"],
    "sample_mutation_features.parquet": ["sample"],
    "sample_mutation_features_rollup.parquet": ["sample", "--include-cell-rollup"],
    "mutation_encoded.parquet": ["enc3"],
    "gene_mutated_matrix.parquet": ["gene", "--kind", "mutated"],
    "parsed_mutation_tokens.parquet": ["parsed-tokens"],
}

for split in ("train", "test"):
    for suffix, command in FEATURE_COMMANDS.items():
        output = PROCESS_DIR / f"{split}_{suffix}"
        if output.is_file():
            continue
        cmd = [
            sys.executable,
            "scripts/make_features.py",
            command[0],
            "--split",
            split,
            "--input",
            str(RAW_SOURCE / f"{split}.csv"),
            "--output",
            str(output),
            *command[1:],
        ]
        print("RUN:", " ".join(cmd))
        subprocess.run(cmd, check=True)

fold_path = PROCESS_DIR / "train_folds.parquet"
if not fold_path.is_file():
    subprocess.run(
        [
            sys.executable,
            "scripts/make_folds.py",
            "--input",
            str(RAW_SOURCE / "train.csv"),
            "--out",
            str(fold_path),
            "--n-splits",
            str(N_SPLITS),
            "--seed",
            str(SEED),
        ],
        check=True,
    )

expected = [
    PROCESS_DIR / f"{split}_{suffix}"
    for split in ("train", "test")
    for suffix in FEATURE_COMMANDS
] + [fold_path]
missing = [str(path) for path in expected if not path.is_file()]
if missing:
    raise FileNotFoundError("생성 후에도 누락된 파일:\n" + "\n".join(missing))
print(f"feature check OK: {len(expected)} files")

## 4. C00~C03 등록과 중복 feature 감사

구성은 요청한 식 그대로 유지한다. `duplicate_fe25_columns`는 모델 행렬에 같은 이름과
값으로 두 번 들어가는 fe25 컬럼 수다.

In [ ]:
import importlib.util
import json
import time
from collections import OrderedDict
from contextlib import redirect_stdout
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import scipy
import sklearn


def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"cannot import {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


train_linear = load_module(
    "notebook_train_linear_final4",
    REPO_DIR / "scripts/train_linear.py",
)
tg = train_linear.load_tabular_driver()

CASES = OrderedDict(
    [
        (
            "C00_rollup16_lsvd",
            ("domain", "rollup16", "enc3", "ptok", "lsvd", "csig", "fe25"),
        ),
        (
            "C01_rollup_lsvd",
            ("domain", "rollup", "enc3", "ptok", "lsvd", "csig", "fe25"),
        ),
        (
            "C02_rollup16_lnmf",
            ("domain", "rollup16", "enc3", "ptok", "lnmf", "csig", "fe25"),
        ),
        (
            "C03_rollup_lnmf",
            ("domain", "rollup", "enc3", "ptok", "lnmf", "csig", "fe25"),
        ),
    ]
)

DUPLICATE_FE25 = {
    "C00_rollup16_lsvd": 9,
    "C01_rollup_lsvd": 25,
    "C02_rollup16_lnmf": 9,
    "C03_rollup_lnmf": 25,
}

for name, blocks in CASES.items():
    unknown = sorted(set(blocks) - set(tg.BLOCK_DESC))
    if unknown:
        raise RuntimeError(f"{name} unknown blocks: {unknown}")
    tg.CONFIGS[name] = {
        "blocks": tuple(blocks),
        "weight": "balanced",
        "desc": f"LogReg final4: {' + '.join(blocks)}",
    }

assert len(CASES) == 4
assert all("fe25" in blocks for blocks in CASES.values())
assert all(
    ("rollup" in blocks) ^ ("rollup16" in blocks)
    for blocks in CASES.values()
)

case_table = pd.DataFrame(
    [
        {
            "case": name,
            "rollup_axis": "rollup16" if "rollup16" in blocks else "rollup",
            "latent_axis": "lnmf" if "lnmf" in blocks else "lsvd",
            "duplicate_fe25_columns": DUPLICATE_FE25[name],
            "blocks": " + ".join(blocks),
        }
        for name, blocks in CASES.items()
    ]
)
display(case_table)
print(f"{len(CASES)} candidates × {len(CVS)} CV = {len(CASES) * len(CVS)} runs")

## 5. 공용 Dataset과 후보별 독립 C 보정 설정

feature builder와 outer folds는 공유하지만, C는 후보마다 자신의 outer-train 내부에서
독립적으로 고른다. 따라서 C01의 중복 구조도 C01에 맞는 규제로 보정된다.

In [ ]:
DRY_RUN = False
RUN_SUBMISSION = True
RESUME_COMPLETED_CASES = True

ARTIFACTS = OUTPUT_ROOT / "artifacts"
STATE_PATH = OUTPUT_ROOT / "suite_state.json"
FAILURE_PATH = OUTPUT_ROOT / "failed_cases.json"
REPORT_DIR = OUTPUT_ROOT / "reports"
CONSOLE_DIR = ARTIFACTS / "console"
for directory in (ARTIFACTS, REPORT_DIR, CONSOLE_DIR):
    directory.mkdir(parents=True, exist_ok=True)

tg.RAW_DIR = RAW_SOURCE
tg.PROC_DIR = PROCESS_DIR
tg.ARTIFACTS = ARTIFACTS

all_needed = set().union(*(set(blocks) for blocks in CASES.values()))
data = tg.Dataset(all_needed, n_splits=N_SPLITS)

linear_config = train_linear.load_yaml(REPO_DIR / "configs/linear.yaml")
notebook_cli = [
    "--configs",
    ",".join(CASES),
    "--cv",
    "all",
    "--n-splits",
    str(N_SPLITS),
    "--seed",
    str(SEED),
    "--tag",
    TAG,
    "--artifacts",
    str(ARTIFACTS),
    "--raw-dir",
    str(RAW_SOURCE),
    "--process-dir",
    str(PROCESS_DIR),
    "--calibration-config",
    "C00_rollup16_lsvd",
    "--c-grid",
    ",".join(map(str, C_GRID)),
    "--inner-valid-fraction",
    str(INNER_VALID_FRACTION),
    "--solver",
    SOLVER,
    "--scaler",
    SCALER,
    "--max-iter",
    str(MAX_ITER),
    "--tol",
    str(TOL),
    "--submission" if RUN_SUBMISSION else "--no-submission",
]
linear_args, forwarded = train_linear.build_parser(linear_config).parse_known_args(
    notebook_cli
)
driver_args = train_linear.driver_args(tg, linear_args, forwarded)
driver_args.topk = TOPK
driver_args.dry_run = DRY_RUN

manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_branch": branch_name,
    "git_commit": commit_sha,
    "raw_source": str(RAW_SOURCE),
    "process_dir": str(PROCESS_DIR),
    "output_root": str(OUTPUT_ROOT),
    "python": platform.python_version(),
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "scikit_learn": sklearn.__version__,
    "cv": list(CVS),
    "n_splits": N_SPLITS,
    "seed": SEED,
    "topk": TOPK,
    "model": {
        "solver": SOLVER,
        "scaler": SCALER,
        "max_iter": MAX_ITER,
        "tol": TOL,
    },
    "calibration": {
        "config": "each_candidate_itself",
        "c_grid": C_GRID,
        "inner_valid_fraction": INNER_VALID_FRACTION,
        "outer_validation_used": False,
    },
    "cases": {name: list(blocks) for name, blocks in CASES.items()},
    "duplicate_fe25_columns": DUPLICATE_FE25,
}
train_linear.atomic_json(OUTPUT_ROOT / "manifest.json", manifest)
print("Dataset/config ready")



## 6. 저장·resume 함수

후보별 C 보정은 해당 후보 실행 안에서 일어난다. 완료된 `(CV, case)`는 다시 보정하거나
학습하지 않고 저장된 artifact를 재사용한다.

In [ ]:
class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, value):
        for stream in self.streams:
            stream.write(value)
        return len(value)

    def flush(self):
        for stream in self.streams:
            stream.flush()


def load_json(path: Path, default):
    if not path.is_file():
        return default
    with path.open(encoding="utf-8") as handle:
        return json.load(handle)


def state_key(cv: str, case_name: str) -> str:
    return f"{cv}::{case_name}"


def result_marker(result: dict) -> dict:
    stem = result["stem"]
    return {
        "cv": result["cv"],
        "case": result["config"],
        "stem": stem,
        "blocks": result["blocks"],
        "duplicate_fe25_columns": DUPLICATE_FE25[result["config"]],
        "n_features_per_fold": result["n_features_per_fold"],
        "oof_macro_f1": result["oof_macro_f1"],
        "oof_macro_f1_singleton": result["oof_macro_f1_singleton"],
        "oof_accuracy": result["oof_accuracy"],
        "fold_macro_f1": result["fold_macro_f1"],
        "elapsed_seconds": result["elapsed_seconds"],
        "per_class_f1": result["per_class_f1"],
        "fold_models": result["fold_models"],
        "selected_C_per_fold": result["linear_calibration"]["selected_C_per_fold"],
        "log_path": str(ARTIFACTS / "logs" / f"{stem}.json"),
        "oof_path": str(ARTIFACTS / "oof" / f"oof_{stem}.csv"),
        "test_path": str(ARTIFACTS / "test_predictions" / f"test_{stem}.csv"),
        "submission_path": str(
            ARTIFACTS / "submissions" / f"submission_{stem}.csv"
        ),
        "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    }


def save_result(result: dict, state: dict, cv: str, trainer) -> None:
    result["linear_calibration"] = trainer.summary(cv)
    result["duplicate_fe25_columns"] = DUPLICATE_FE25[result["config"]]
    result["runtime_versions"] = {
        "python": platform.python_version(),
        "numpy": np.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
    }
    train_linear.atomic_json(
        ARTIFACTS / "logs" / f"{result['stem']}.json",
        result,
    )
    state[state_key(cv, result["config"])] = result_marker(result)
    train_linear.atomic_json(STATE_PATH, state)


def marker_is_complete(marker: dict) -> bool:
    required = ["log_path", "oof_path", "test_path"]
    if RUN_SUBMISSION:
        required.append("submission_path")
    return all(Path(marker.get(key, "")).is_file() for key in required)


state = load_json(STATE_PATH, {})

## 7. C00~C03 × SKF/Group5 실행

각 실행은 `calibration_config=case_name`으로 새 trainer를 만들기 때문에 후보별 C가
독립적으로 선택된다.

In [ ]:
failures = load_json(FAILURE_PATH, {})
suite_started = time.perf_counter()
total_runs = len(CVS) * len(CASES)
run_index = 0

for cv in CVS:
    for case_name in CASES:
        run_index += 1
        key = state_key(cv, case_name)
        existing = state.get(key)
        if (
            RESUME_COMPLETED_CASES
            and existing is not None
            and marker_is_complete(existing)
        ):
            print(f"[{run_index:02d}/{total_runs}] SKIP {cv} / {case_name}")
            continue

        print(f"\n[{run_index:02d}/{total_runs}] START {cv} / {case_name}")
        trainer = train_linear.LinearFoldTrainer(
            calibration_config=case_name,
            c_grid=C_GRID,
            inner_valid_fraction=INNER_VALID_FRACTION,
            solver=SOLVER,
            scaler=SCALER,
            max_iter=MAX_ITER,
            tol=TOL,
            seed=SEED,
            n_splits=N_SPLITS,
        )
        trainer.begin_config(config=case_name, cv=cv)
        console_path = CONSOLE_DIR / f"{cv}_{case_name}.log"
        try:
            with console_path.open("w", encoding="utf-8") as log_handle:
                with redirect_stdout(Tee(sys.stdout, log_handle)):
                    result = tg.run_config(
                        data,
                        config=case_name,
                        cv=cv,
                        args=driver_args,
                        fit_model=trainer,
                    )
            result["linear_calibration"] = trainer.summary(cv)
            save_result(result, state, cv, trainer)
            failures.pop(key, None)
            train_linear.atomic_json(FAILURE_PATH, failures)
            print(
                f"[{run_index:02d}/{total_runs}] DONE {cv} / {case_name}: "
                f"F1={result['oof_macro_f1']:.6f}, "
                f"C={trainer.selected_c[cv]}"
            )
        except Exception as exc:
            failures[key] = {
                "cv": cv,
                "case": case_name,
                "type": type(exc).__name__,
                "message": str(exc),
                "console_path": str(console_path),
                "failed_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            train_linear.atomic_json(FAILURE_PATH, failures)
            print(
                f"[{run_index:02d}/{total_runs}] FAILED {cv} / {case_name}: "
                f"{type(exc).__name__}: {exc}"
            )

print(f"suite elapsed: {(time.perf_counter() - suite_started) / 3600:.2f} h")
print(f"completed: {len(state)}/{total_runs}, failed: {len(failures)}")

## 8. 비교표·2×2 pairwise delta

C00을 공통 기준으로 보여주고, rollup 교체 효과와 latent 교체 효과는 동일한 상대 축끼리
별도 계산한다.

In [ ]:
state = load_json(STATE_PATH, {})
reference_case = "C00_rollup16_lsvd"
missing_references = [
    cv for cv in CVS if state_key(cv, reference_case) not in state
]
if missing_references:
    raise RuntimeError(f"C00 result가 없는 CV: {missing_references}")

rows = []
for cv in CVS:
    reference_f1 = state[state_key(cv, reference_case)]["oof_macro_f1"]
    for case_name in CASES:
        marker = state.get(state_key(cv, case_name))
        if marker is None:
            continue
        fold_models = marker.get("fold_models", [])
        converged = sum(bool(model.get("converged")) for model in fold_models)
        axis = case_table.set_index("case").loc[case_name]
        rows.append(
            {
                "cv": "SKF" if cv == "skf" else "Group5",
                "case": case_name,
                "rollup_axis": axis["rollup_axis"],
                "latent_axis": axis["latent_axis"],
                "duplicate_fe25_columns": marker["duplicate_fe25_columns"],
                "features_mean": np.mean(marker["n_features_per_fold"]),
                "oof_macro_f1": marker["oof_macro_f1"],
                "delta_vs_C00": marker["oof_macro_f1"] - reference_f1,
                "singleton_f1": marker["oof_macro_f1_singleton"],
                "accuracy": marker["oof_accuracy"],
                "fold_mean": np.mean(marker["fold_macro_f1"]),
                "fold_std": np.std(marker["fold_macro_f1"]),
                "selected_C": ",".join(map(str, marker["selected_C_per_fold"])),
                "converged_folds": f"{converged}/{N_SPLITS}",
                "elapsed_min": marker["elapsed_seconds"] / 60,
            }
        )

summary = pd.DataFrame(rows).sort_values(
    ["cv", "oof_macro_f1"], ascending=[True, False]
)
summary.to_csv(REPORT_DIR / "logreg_final4_summary.csv", index=False)
summary.to_json(
    REPORT_DIR / "logreg_final4_summary.json",
    orient="records",
    force_ascii=False,
    indent=2,
)


def color_delta(value):
    if value > 0:
        return "color: #137333; background-color: #e6f4ea"
    if value < 0:
        return "color: #c5221f; background-color: #fce8e6"
    return "font-weight: bold"


styled = (
    summary.style.format(
        {
            "features_mean": "{:,.0f}",
            "oof_macro_f1": "{:.6f}",
            "delta_vs_C00": "{:+.6f}",
            "singleton_f1": "{:.6f}",
            "accuracy": "{:.6f}",
            "fold_mean": "{:.6f}",
            "fold_std": "{:.6f}",
            "elapsed_min": "{:.1f}",
        }
    )
    .map(color_delta, subset=["delta_vs_C00"])
    .hide(axis="index")
)
display(styled)
(REPORT_DIR / "logreg_final4_summary.html").write_text(
    styled.to_html(), encoding="utf-8"
)

score = {
    (row["cv"], row["case"]): row["oof_macro_f1"]
    for row in rows
}
pairwise_rows = []
comparisons = {
    "rollup effect with lsvd": ("C01_rollup_lsvd", "C00_rollup16_lsvd"),
    "rollup effect with lnmf": ("C03_rollup_lnmf", "C02_rollup16_lnmf"),
    "lnmf effect with rollup16": ("C02_rollup16_lnmf", "C00_rollup16_lsvd"),
    "lnmf effect with rollup": ("C03_rollup_lnmf", "C01_rollup_lsvd"),
}
for cv in ("SKF", "Group5"):
    for comparison, (candidate, reference) in comparisons.items():
        if (cv, candidate) in score and (cv, reference) in score:
            pairwise_rows.append(
                {
                    "cv": cv,
                    "comparison": comparison,
                    "candidate": candidate,
                    "reference": reference,
                    "macro_f1_delta": score[(cv, candidate)] - score[(cv, reference)],
                }
            )
pairwise = pd.DataFrame(pairwise_rows)
pairwise.to_csv(REPORT_DIR / "logreg_final4_pairwise_delta.csv", index=False)
display(
    pairwise.style.format({"macro_f1_delta": "{:+.6f}"})
    .map(color_delta, subset=["macro_f1_delta"])
    .hide(axis="index")
)

print("reports:", REPORT_DIR)

## 9. 전체 산출물 ZIP



In [ ]:
archive_base = WORK_ROOT / f"{TAG}_{commit_sha[:8]}"
archive_path = Path(
    shutil.make_archive(str(archive_base), "zip", root_dir=OUTPUT_ROOT)
)
print("archive:", archive_path)
print("size MB:", archive_path.stat().st_size / 1024**2)

if IN_COLAB:
    from google.colab import files

    print("필요하면 다음 줄의 주석을 풀어 다운로드하세요.")
    print(f"# files.download({str(archive_path)!r})")
